# Cac dinh nghia can biet
* Sample Rate: So lan lay mau trong mot giay
* Bit Depth: So bit dung de luu moi sample
* Chanel:
    * Mono:1 chanel
    * Stereo: 2 chanel
    -> can chuyen ve mono
* Amplitude(Bien do): Do lon cua song am
* Decibel: Don vi do cuong do am thanh
* Waveform: Bieu dien am thanh theo thoi gian
### Pipeline
crawl audio -> demus-> vocal-> normalized-> segment.wav ->subprocess -> metadata.csv -> ..


In [ ]:
import os
import gc
import torch
import yt_dlp
import subprocess
from pydub import AudioSegment, silence
from faster_whisper import WhisperModel

# cau hinh
BASE_DIR = "/home/minwell/Documents/python project/voice_clone/data"
RAW_DIR = os.path.join(BASE_DIR, "raw/voice_raw")
VOCAL_DIR = os.path.join(BASE_DIR, "raw/vocals_only")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")

PLAYLIST_URL = "https://youtube.com/playlist?list=PL_9p5d0_uJVIGA5j0YvOkVLfzn0j_8IY_&si=Bpn8f-lkHpF0Ap51"

for d in [RAW_DIR, VOCAL_DIR, PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

#dowload
def download_playlist(url):
    print("🎬 Bước 1: Đang tải videotừ Playlist...")
    
    ydl_opts = {
        "format": "bestaudio/best",
        "noplaylist": False,
        "ignoreerrors": True,
        "playlist_items": "1-20", 
        
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "wav",
                "preferredquality": "192",
            }
        ],
        "outtmpl": os.path.join(RAW_DIR, "%(playlist_index)s-%(title)s.%(ext)s"),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

# -tach vocal bang demucs
def separate_vocals_batch():
    print("🎤 Bước 2: Demucs tách vocal hàng loạt...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    files_to_process = [f for f in os.listdir(RAW_DIR) if f.endswith(".wav")]
    
    for i, audio_file in enumerate(files_to_process):
        print(f"📦 Đang tách file {i+1}/{len(files_to_process)}: {audio_file}")
        input_path = os.path.join(RAW_DIR, audio_file)
      
        folder_name = os.path.splitext(audio_file)[0]
        check_path = os.path.join(VOCAL_DIR, "htdemucs", folder_name, "vocals.wav")
        
        if os.path.exists(check_path):
            print(f"⏩ Đã tồn tại, bỏ qua: {audio_file}")
            continue

        try:
            subprocess.run([
                "demucs", "--two-stems=vocals", "-d", device, 
                "-o", VOCAL_DIR, input_path
            ], check=True)
        except Exception as e:
            print(f"❌ Demucs lỗi {audio_file}: {e}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# SEGMENTATION 
def process_audio_batch():
    print("✂️ Bước 3: Cắt nhỏ toàn bộ các bản Vocal...")
    model_name = "htdemucs"
    vocal_root = os.path.join(VOCAL_DIR, model_name)
    
    if not os.path.exists(vocal_root):
        print("❌ Không tìm thấy thư mục vocals. Kiểm tra lại Bước 2.")
        return

    for root, dirs, files in os.walk(vocal_root):
        for file in files:
            if file != "vocals.wav":
                continue

            parent_name = os.path.basename(root) 
            path = os.path.join(root, file)
           
            if any(f.startswith(parent_name) for f in os.listdir(PROCESSED_DIR)):
                print(f"⏩ Đã cắt: {parent_name}, bỏ qua.")
                continue

            print(f"🔪 Đang cắt: {parent_name}")
            audio = AudioSegment.from_wav(path).set_channels(1)
            segments = silence.split_on_silence(audio, min_silence_len=300, silence_thresh=-40)
            
            valid = [s for s in segments if 3000 <= len(s) <= 10000]
            for i, seg in enumerate(valid[:200]):
                name = f"{parent_name}_seg_{i:03d}.wav"
                seg.export(os.path.join(PROCESSED_DIR, name), format="wav")

#sub
def start_transcription_batch():
    print("🎙️ Bước 4: Whisper gán nhãn hàng loạt...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model = WhisperModel("medium", device=device, compute_type="float16" if device == "cuda" else "int8")
    
    wav_files = sorted(f for f in os.listdir(PROCESSED_DIR) if f.endswith(".wav"))
    metadata_path = os.path.join(PROCESSED_DIR, "metadata.csv")

    existing_wavs = set()
    if os.path.exists(metadata_path):
        with open(metadata_path, "r", encoding="utf-8") as f:
            for line in f:
                existing_wavs.add(line.split("|")[0])

    for wav in wav_files:
        if wav in existing_wavs:
            continue

        path = os.path.join(PROCESSED_DIR, wav)
        try:
            segments, _ = model.transcribe(path, beam_size=3, language="vi")
            text = " ".join(seg.text.strip() for seg in segments).strip()
            
            if not text:
                continue

            with open(metadata_path, "a", encoding="utf-8") as f:
                f.write(f"{wav}|{text}\n")
            print(f"✔ {wav}")
        except Exception as e:
            print(f"❌ Whisper lỗi {wav}: {e}")

if __name__ == "__main__":
    download_playlist(PLAYLIST_URL)
    
    separate_vocals_batch()
    
    process_audio_batch()
    
    start_transcription_batch()
    
print("done")

🎬 Bước 1: Đang tải 20 video đầu tiên từ Playlist...
[youtube:tab] Extracting URL: https://youtube.com/playlist?list=PL_9p5d0_uJVIGA5j0YvOkVLfzn0j_8IY_&si=Bpn8f-lkHpF0Ap51
[youtube:tab] PL_9p5d0_uJVIGA5j0YvOkVLfzn0j_8IY_: Downloading webpage
[youtube:tab] PL_9p5d0_uJVIGA5j0YvOkVLfzn0j_8IY_: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: Radio Văn Học
[youtube:tab] Playlist Radio Văn Học: Downloading 20 items of 74
[download] Downloading item 1 of 20
[youtube] Extracting URL: https://www.youtube.com/watch?v=dfi5bO6RJVo
[youtube] dfi5bO6RJVo: Downloading webpage


[youtube] dfi5bO6RJVo: Downloading android vr player API JSON
[info] dfi5bO6RJVo: Downloading 1 format(s): 251
[download] Destination: /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/01-RADIO VĂN HỌC #1： HÌNH TƯỢNG NGƯỜI LÍNH TRONG ＂TÂY TIẾN＂ VÀ ＂VIỆT BẮC＂.webm
[download] 100% of    8.90MiB in 00:00:00 at 9.35MiB/s   
[ExtractAudio] Destination: /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/01-RADIO VĂN HỌC #1： HÌNH TƯỢNG NGƯỜI LÍNH TRONG ＂TÂY TIẾN＂ VÀ ＂VIỆT BẮC＂.wav
Deleting original file /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/01-RADIO VĂN HỌC #1： HÌNH TƯỢNG NGƯỜI LÍNH TRONG ＂TÂY TIẾN＂ VÀ ＂VIỆT BẮC＂.webm (pass -k to keep)
[download] Downloading item 2 of 20
[youtube] Extracting URL: https://www.youtube.com/watch?v=3EGYY76Mz_I
[youtube] 3EGYY76Mz_I: Downloading webpage


[youtube] 3EGYY76Mz_I: Downloading android vr player API JSON
[info] 3EGYY76Mz_I: Downloading 1 format(s): 251
[download] Destination: /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/02-RADIO VĂN HỌC #2： NỖI NHỚ TRONG ＂TÂY TIẾN＂ VÀ ＂VIỆT BẮC＂.webm
[download] 100% of   19.76MiB in 00:00:00 at 22.17MiB/s    
[ExtractAudio] Destination: /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/02-RADIO VĂN HỌC #2： NỖI NHỚ TRONG ＂TÂY TIẾN＂ VÀ ＂VIỆT BẮC＂.wav
Deleting original file /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/02-RADIO VĂN HỌC #2： NỖI NHỚ TRONG ＂TÂY TIẾN＂ VÀ ＂VIỆT BẮC＂.webm (pass -k to keep)
[download] Downloading item 3 of 20
[youtube] Extracting URL: https://www.youtube.com/watch?v=vAHn-frJGaI
[youtube] vAHn-frJGaI: Downloading webpage


[youtube] vAHn-frJGaI: Downloading android vr player API JSON
[info] vAHn-frJGaI: Downloading 1 format(s): 251
[download] Destination: /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/03-RADIO VĂN HỌC #3： TUỔI TRẺ NÊN VỘI VÃ HAY ＂CHẬM LẠI MỘT CHÚT＂？.webm
[download] 100% of   19.47MiB in 00:00:01 at 13.81MiB/s    
[ExtractAudio] Destination: /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/03-RADIO VĂN HỌC #3： TUỔI TRẺ NÊN VỘI VÃ HAY ＂CHẬM LẠI MỘT CHÚT＂？.wav
Deleting original file /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/03-RADIO VĂN HỌC #3： TUỔI TRẺ NÊN VỘI VÃ HAY ＂CHẬM LẠI MỘT CHÚT＂？.webm (pass -k to keep)
[download] Downloading item 4 of 20
[youtube] Extracting URL: https://www.youtube.com/watch?v=4ehFLD12LrI
[youtube] 4ehFLD12LrI: Downloading webpage


[youtube] 4ehFLD12LrI: Downloading android vr player API JSON
[info] 4ehFLD12LrI: Downloading 1 format(s): 251
[download] Destination: /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/04-RADIO VĂN HỌC #4： KHÁT KHAO TÌNH YÊU TRONG ＂SÓNG＂.webm
[download] 100% of   22.23MiB in 00:00:01 at 11.56MiB/s    
[ExtractAudio] Destination: /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/04-RADIO VĂN HỌC #4： KHÁT KHAO TÌNH YÊU TRONG ＂SÓNG＂.wav
Deleting original file /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/04-RADIO VĂN HỌC #4： KHÁT KHAO TÌNH YÊU TRONG ＂SÓNG＂.webm (pass -k to keep)
[download] Downloading item 5 of 20
[youtube] Extracting URL: https://www.youtube.com/watch?v=pWZip2Lyj-4
[youtube] pWZip2Lyj-4: Downloading webpage


...

📦 Đang tách file 19/20: 14-RADIO VĂN HỌC #15： Ý NGHĨA CỦA NHỮNG CUỘC CHIA LY.wav
Important: the default model was recently changed to `htdemucs` the latest Hybrid Transformer Demucs model. In some cases, this model can actually perform worse than previous models. To get back the old default model use `-n mdx_extra_q`.
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /home/minwell/Documents/python project/voice_clone/data/raw/vocals_only/htdemucs
Separating track /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/14-RADIO VĂN HỌC #15： Ý NGHĨA CỦA NHỮNG CUỘC CHIA LY.wav


100%|██████████████████████████████████████████████| 1134.8999999999999/1134.8999999999999 [00:38<00:00, 29.76seconds/s]


📦 Đang tách file 20/20: 08-RADIO VĂN HỌC #8： GIÁ TRỊ THỨC TỈNH CỦA ＂BÁT CHÁO HÀNH＂ VÀ ＂ẤM NƯỚC ĐẦY VÀ HÃY CÒN ẤM＂.wav
Important: the default model was recently changed to `htdemucs` the latest Hybrid Transformer Demucs model. In some cases, this model can actually perform worse than previous models. To get back the old default model use `-n mdx_extra_q`.
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /home/minwell/Documents/python project/voice_clone/data/raw/vocals_only/htdemucs
Separating track /home/minwell/Documents/python project/voice_clone/data/raw/voice_raw/08-RADIO VĂN HỌC #8： GIÁ TRỊ THỨC TỈNH CỦA ＂BÁT CHÁO HÀNH＂ VÀ ＂ẤM NƯỚC ĐẦY VÀ HÃY CÒN ẤM＂.wav


100%|██████████████████████████████████████████████████████████████████████| 848.25/848.25 [00:28<00:00, 29.69seconds/s]


✂️ Bước 3: Cắt nhỏ toàn bộ các bản Vocal...
🔪 Đang cắt: 18-Radio Đời sống： Không có một đam mê lớn lao là thất bại？
🔪 Đang cắt: 08-RADIO VĂN HỌC #8： GIÁ TRỊ THỨC TỈNH CỦA ＂BÁT CHÁO HÀNH＂ VÀ ＂ẤM NƯỚC ĐẦY VÀ HÃY CÒN ẤM＂
🔪 Đang cắt: 04-RADIO VĂN HỌC #4： KHÁT KHAO TÌNH YÊU TRONG ＂SÓNG＂
🔪 Đang cắt: 12-RADIO VĂN HỌC #12： TÔ HOÀI VIẾT VỀ NGUYỄN TUÂN
🔪 Đang cắt: 20-Radio Văn học： Bến quê - Nguyễn Minh Châu
🔪 Đang cắt: 11-RADIO VĂN HỌC #11： XUÂN QUỲNH - CHUYỆN ĐỜI, CHUYỆN YÊU
🔪 Đang cắt: 19-Radio Văn học： Tản văn “Khúc ba mươi” - Nguyễn Ngọc Tư
🔪 Đang cắt: 15-RADIO VĂN HỌC #16： KHÁT VỌNG SỐNG CỦA MỊ TRONG ＂VỢ CHỒNG A PHỦ＂
🔪 Đang cắt: 01-RADIO VĂN HỌC #1： HÌNH TƯỢNG NGƯỜI LÍNH TRONG ＂TÂY TIẾN＂ VÀ ＂VIỆT BẮC＂
🔪 Đang cắt: 02-RADIO VĂN HỌC #2： NỖI NHỚ TRONG ＂TÂY TIẾN＂ VÀ ＂VIỆT BẮC＂
🔪 Đang cắt: 10-RADIO VĂN HỌC #10： NHÀ THƠ QUANG DŨNG - NGƯỜI VIẾT SỬ TÂY TIẾN
🔪 Đang cắt: 17-RADIO VĂN HỌC #18： Tấm lòng học trò và bài học đáng giá từ người thầy Chế Lan Viên 👩‍🏫
🔪 Đang cắt: 07-RADIO VĂN HỌC #7： HÌNH TƯỢ